# Gemma3 4B — KNF/Natural Farming SFT Fine-Tuning

**Runtime:** Google Colab T4 (free tier, 16GB VRAM)  
**Model:** `unsloth/gemma-3-4b-it` — 4B text-only, fits T4 with 4-bit QLoRA  
**Dataset:** `merged_sharegpt.jsonl` hosted on HuggingFace Datasets (pre-normalized)  
**Output:** LoRA adapter + GGUF (q8_0) pushed to HuggingFace Hub

> Fallback: change `MODEL_NAME` to `unsloth/gemma-3-1b-it` if 4B still OOMs

---
**Before running:** set your HuggingFace username in the config cell.

## 1. Install dependencies

In [ ]:
!pip install --quiet unsloth trl datasets transformers bitsandbytes peft accelerate huggingface_hub

## 2. Configuration

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────
HF_USERNAME     = "CopyleftCultivars"
HF_DATASET_REPO = f"{HF_USERNAME}/knf-sharegpt"
HF_ADAPTER_REPO = f"{HF_USERNAME}/gemma3-natural-farming-lora"
HF_GGUF_REPO    = f"{HF_USERNAME}/gemma3-natural-farming-gguf"
# ─────────────────────────────────────────────────────────────────────────

# gemma-3-4b-it: 4B text-only model, comfortably fits T4 with 4-bit QLoRA
# Swap to unsloth/gemma-3-1b-it if you hit OOM (unlikely at 4B)
MODEL_NAME      = "unsloth/gemma-3-4b-it"
MAX_SEQ_LENGTH  = 2048
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
BATCH_SIZE      = 2
GRAD_ACCUM      = 4    # effective batch = 8
WARMUP_STEPS    = 100
NUM_EPOCHS      = 3
LEARNING_RATE   = 2e-4
OUTPUT_DIR      = "outputs"
ADAPTER_DIR     = "gemma3-knf-adapter"
GGUF_DIR        = "gemma3-knf-gguf"

print("Config loaded.")
print(f"  Model        : {MODEL_NAME}")
print(f"  Dataset repo : {HF_DATASET_REPO}")
print(f"  Adapter repo : {HF_ADAPTER_REPO}")
print(f"  GGUF repo    : {HF_GGUF_REPO}")

## 3. HuggingFace login

In [ ]:
from huggingface_hub import login, HfApi

# Colab Secrets (recommended): add HF_TOKEN in the 🔑 sidebar
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # fallback: manual token prompt

## 4. Load model with 4-bit QLoRA via Unsloth

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

# Reduces allocator fragmentation — helps on tight VRAM budgets
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded and LoRA adapters attached.")

## 5. Load dataset from HuggingFace

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_DATASET_REPO, data_files="train.jsonl", split="train")
print(f"Dataset loaded: {len(ds):,} records")
print("Sample record:")
print(ds[0])

## 6. Format with chat template

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="gemma-it")

# Dataset is pre-normalized: strict human/gpt alternation, no system turns.
# standardize_sharegpt maps human→user, gpt→assistant for the tokenizer.
ds = standardize_sharegpt(ds)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

ds = ds.map(formatting_prompts_func, batched=True)
print(f"Formatting done. Training examples: {len(ds):,}")
print(ds[0]["text"][:500])

## 7. Train

In [ ]:
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,
        bf16=False,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining complete. Runtime: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 8. Save LoRA adapter and push to HuggingFace Hub

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved locally to '{ADAPTER_DIR}'")

model.push_to_hub(HF_ADAPTER_REPO)
tokenizer.push_to_hub(HF_ADAPTER_REPO)
print(f"Adapter pushed to https://huggingface.co/{HF_ADAPTER_REPO}")

## 9. Export GGUF (q8_0) for CPU inference

q8_0 gives better accuracy than q4_k_m at ~2x file size.  
Used by the HuggingFace Spaces chatbot via `llama-cpp-python`.

In [ ]:
import os

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q8_0",
)
print(f"GGUF saved to '{GGUF_DIR}/'")

for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
import glob
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HF_GGUF_REPO, repo_type="model", exist_ok=True, private=False)

for gguf_path in glob.glob(f"{GGUF_DIR}/*.gguf"):
    filename = os.path.basename(gguf_path)
    api.upload_file(
        path_or_fileobj=gguf_path,
        path_in_repo=filename,
        repo_id=HF_GGUF_REPO,
        repo_type="model",
    )
    print(f"Uploaded {filename} → https://huggingface.co/{HF_GGUF_REPO}")

## Done!

| Artifact | Location |
|---|---|
| LoRA adapter | `https://huggingface.co/{HF_ADAPTER_REPO}` |
| GGUF model   | `https://huggingface.co/{HF_GGUF_REPO}` |

Next: update `GGUF_REPO` in `spaces/app.py` to point at `HF_GGUF_REPO` and deploy to HuggingFace Spaces.